In [ ]:
import pymupdf
import json
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field, asdict
from typing import Optional, Self, Iterator, Any
import tiktoken
from pathlib import Path
import re
from PIL import Image
from IPython.display import display
import yaml

from class_objects import DocumentData
from partition_processing import process_partitions, save_images

%matplotlib widget

In [ ]:
def load_config(config: Path|str) -> dict:
    """Load YAML configuration file."""
    with open(config, 'r') as file:
        return yaml.safe_load(file)

In [ ]:
# document_name = 'sofimshc_1-1'
# document_name = 'ns-en-1992-1-1_2004+a1_2014+na_2024_en_002'
# document_name = 'ns-en-1993-1-3_2006+na_2015_en_001'
document_name = 'ns-en-1995-1-1_2004+a2_2014+na_2024_en_001'

In [ ]:
config = load_config('../volumes/configs/partitionConfig.yaml')

pdf_path = Path(config['pdf_dir_path'])
output_path = Path(config['output_dir_path']) / document_name

config

In [ ]:
output_path.mkdir(parents=True, exist_ok=True)

In [ ]:
doc = pymupdf.open(pdf_path / f'{document_name}.pdf')

In [ ]:
# rect = pymupdf.Rect(config['pdf_partition_config']['crop'])
rect = pymupdf.Rect(38, 50, 563, 785)  # A4 size in points

rect
page = doc[40]
page.set_cropbox(rect)

pix = page.get_pixmap()
image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

display(image)

In [ ]:
partition = DocumentData(json_path=output_path / 'unstructured_partitions_hi_res.json')

len(partition)

In [ ]:
part = partition
crop = np.array(config['pdf_partition_config']['crop'])

cropped_partitions = []
for p in part:
    page_number = p.metadata.page_number
    page = doc[page_number - 1]
    scale = p.metadata.coordinates.layout_height / page.rect.height

    crop_scaled = crop * scale

    coordinates = np.array(p.metadata.coordinates.points)
    if not (
        coordinates.max(axis=0)[0] <= crop_scaled[0] or
        coordinates.min(axis=0)[0] >= crop_scaled[2] or
        coordinates.min(axis=0)[1] <= crop_scaled[1] or
        coordinates.max(axis=0)[1] >= crop_scaled[3]
    ):
        cropped_partitions.append(p)

len(cropped_partitions)

In [ ]:
partition.crop_documents(doc, crop)

len(partition)

In [ ]:
start = 54
end = 54

data = process_partitions(partition, doc, start, end)
with open(output_path / 'document_chunks.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

data

In [ ]:
doc = pymupdf.open(pdf_path / f'{document_name}.pdf')
page = doc[53]

crop = np.array([467.50341796875, 3797.4560546875, 2203.2392333333346, 3855.758544921875])
crop *= page.rect.height / partition[0].metadata.coordinates.layout_height

rect = pymupdf.Rect(crop.tolist())
page.set_cropbox(rect)

display(page.get_text())

pix = page.get_pixmap()
image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
display(image)

In [ ]:
sections = []
for item in data:
    if not item['sections'] is sections:
        print(item['sections'][-1])

    sections = item['sections']


In [ ]:
len(data)

In [ ]:
save_images(partition, data, doc, output_path / 'images')